<div style="background: linear-gradient(120deg,#1e1b4b,#312e81); border-radius:16px; padding:36px 40px; color:#e2e8f0;">
<h1 style="margin:0; font-size:2.1em; color:#ffffff;">🌌 Embeddings & Vector Databases</h1>
<h3 style="margin:6px 0 0 0; font-weight:400; color:#93c5fd;">Session 2 of 4 — the geometry of meaning</h3>
<p style="margin-top:18px; color:#cbd5e1;">NordWind Energy Workshop Series · Retrieval, Vectors & Graphs</p>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noctetemp/nordwind-workshop/blob/main/session2_vectors_en.ipynb)

## 🗺️ Today's journey

Last session your retriever did its search with **one matrix multiplication** — comparing the query against *every* chunk. Today we find out what that multiplication really computes, watch it hit a wall, and replace it with real infrastructure.

* **🧭 What an embedding actually is** — meaning as coordinates. Live demo: a paraphrase outscores an unrelated sentence ~17×... and then an *opposite instruction* outscores them all. Similarity ≠ agreement.
* **👁️ Seeing semantic space** — the wow moment: every chunk of NordWind projected into an interactive 2D map. Postmortems cluster with postmortems, runbooks with runbooks — *nobody told the model that*. We'll drop a query into the map and watch its neighbors light up.
* **🧱 The scale wall** — we benchmark brute force, extrapolate to 5 million chunks, and meet **HNSW**: how vector DBs search millions of vectors while touching only a few hundred.
* **🗄️ A real vector database** — Qdrant, running *inside this notebook*. Collections, payloads, and the underrated superpower: **metadata filtering**.
* **🔎 Where pure vectors fumble** — exact identifiers like `ISO 20022`. Fix: **hybrid search** (BM25 + vectors, fused).
* **🎯 Reranking** — a second, smarter model re-reads the top 20 and reorders them. Cheap precision.
* **🏗️ RAG v2** — everything wired together... and one final rerun of *the hard question*. Spoiler: better plumbing, same wall. 😏

> Everything runs on the same NordWind world as Session 1 — same 65 documents, same 20 incidents, same anime engineers.

## 0 · Setup — same world, more tools

In [ ]:
%pip -q install sentence-transformers qdrant-client umap-learn rank_bm25 plotly anthropic tiktoken
print("✅ ready")

In [ ]:
import json, urllib.request, pathlib, os
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/noctetemp/nordwind-workshop/main/dataset/documents.jsonl"
LOCAL = pathlib.Path("documents.jsonl")
if not LOCAL.exists():
    urllib.request.urlretrieve(DATA_URL, LOCAL)
docs = [json.loads(line) for line in LOCAL.read_text().splitlines()]

def chunk_text(text, size=800, overlap=150):
    chunks, i = [], 0
    while i < len(text):
        chunks.append(text[i:i+size]); i += size - overlap
    return chunks

chunks, meta = [], []
for d in docs:
    for j, ch in enumerate(chunk_text(d['text'])):
        chunks.append(ch)
        meta.append({"doc_id": d['doc_id'], "doc_type": d['doc_type'],
                     "title": d['title'], "date": d['date'], "chunk": j})
print(f"📚 {len(docs)} docs → {len(chunks)} chunks (same pipeline as Session 1)")

In [ ]:
# Anthropic client (same pattern as Session 1 — LLM cells degrade gracefully without a key)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass
import anthropic
MODEL = "claude-sonnet-4-6"
client = anthropic.Anthropic() if os.environ.get("ANTHROPIC_API_KEY") else None
def ask_claude(prompt, max_tokens=600):
    if client is None:
        return "[LLM skipped — no API key configured]"
    r = client.messages.create(model=MODEL, max_tokens=max_tokens,
                               messages=[{"role": "user", "content": prompt}])
    return r.content[0].text

---
## 1 · 🧭 What an embedding actually is

An embedding model is a neural network trained on one deceptively simple objective: **sentences that mean similar things should land close together; sentences that don't, shouldn't.** Train that on billions of pairs, and the network is forced to invent a coordinate system for *meaning itself*.

The output for any text is a vector — for our model, **384 numbers**. No single number means anything ("dimension 217 = angriness" is not how it works); meaning lives in the *directions* of the space, distributed across all coordinates.

Closeness is measured with **cosine similarity** — the angle between two vectors: `1.0` = same direction (same meaning), `0` = unrelated, negative = opposing. Because we normalize every vector to length 1, cosine similarity is literally just the dot product. Let's feel the space:

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")   # 384 dims

pairs = [
    ("The payment gateway timed out during peak hours.",
     "Transactions were failing in the evening because the PSP was slow."),   # paraphrase, ~zero shared keywords
    ("The payment gateway timed out during peak hours.",
     "The billing engine computes invoices from meter readings."),            # same domain, different topic
    ("The payment gateway timed out during peak hours.",
     "Frieren finally learned to appreciate her time with Himmel."),          # unrelated
    ("Restart the service to clear the cache.",
     "Do NOT restart the service, the cache must be preserved."),             # ⚠️ opposite instructions!
]
for a, b in pairs:
    va, vb = embedder.encode([a, b], normalize_embeddings=True)
    print(f"{float(va @ vb):5.2f}  |  {a[:44]:44s} ↔  {b[:48]}")

Three lessons hiding in four numbers:

1. The **paraphrase** (~0.5) crushes the unrelated sentence (~0.03) with nearly zero shared keywords — this is what made Session 1's retrieval work. (0.5, not 0.9 — real-world paraphrase scores are lower than people expect; what matters is the *gap*, which is why we rank rather than threshold.)
2. Same-domain-different-topic lands in between — embeddings capture *aboutness*, gradually.
3. ⚠️ Now the shocker: **"restart" vs "do NOT restart" scores ~0.8 — the highest pair on the board, above the true paraphrase.** They're intensely about the same thing, and similarity measures *aboutness* — not agreement, not truth, not logic. Remember this the day someone proposes pure vector search for compliance rules or safety procedures.

Now — 384 dimensions is unvisualizable. Or is it?

---
## 2 · 👁️ Seeing semantic space

We'll embed **every chunk of NordWind** and use UMAP to project 384 dimensions down to 2, preserving neighborhoods as faithfully as possible. Fair warning about the map: distances are distorted (that's unavoidable when crushing 384D→2D) — trust the *clusters*, not the exact distances.

The plot is interactive: **hover** to read chunks, **zoom** into clusters, click legend entries to hide document types.

In [ ]:
E = embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=True)
print("Embedding matrix:", E.shape)

In [ ]:
import umap
import pandas as pd
import plotly.express as px

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42).fit(E)
xy = reducer.embedding_          # keep the fitted reducer — we'll reuse it for the query

df = pd.DataFrame({
    "x": xy[:, 0], "y": xy[:, 1],
    "type": [m["doc_type"] for m in meta],
    "doc": [m["doc_id"] for m in meta],
    "text": [c[:150].replace("\n", " ") for c in chunks]})

fig = px.scatter(df, x="x", y="y", color="type", hover_name="doc", hover_data={"text": True, "x": False, "y": False},
    title="Every chunk of NordWind's knowledge, as a map of meaning",
    color_discrete_map={"postmortem": "#f87171", "runbook": "#34d399",
                        "adr": "#60a5fa", "slack_thread": "#f59e0b", "overview": "#a78bfa"},
    width=950, height=620)
fig.update_traces(marker=dict(size=7, opacity=0.85))
fig.show()

🎤 **Stop and look before scrolling.** The model was never told what a postmortem or a runbook is. It read raw text — and document types form visible regions anyway, because postmortems *talk like* postmortems. Zoom into any mixed area: you'll find a runbook sitting inside postmortem territory because they discuss the **same incident** — topic beats format. That's semantic space.

Now the real magic: retrieval, on this map, is just **proximity**. Let's drop a query in and light up what Session 1's retriever was actually doing:

In [ ]:
query = "Why did payments time out in the evening?"
qv = embedder.encode([query], normalize_embeddings=True)
scores = (E @ qv.T).ravel()
top10 = set(np.argsort(-scores)[:10])

# project the query into the SAME 2D map, reusing the fitted reducer (approximate by nature)
qxy = reducer.transform(qv)

import plotly.graph_objects as go
fig = go.Figure()
fig.add_scatter(x=xy[:,0], y=xy[:,1], mode="markers", name="all chunks",
                marker=dict(size=6, color="#cbd5e1"),
                text=[f"{m['doc_id']}: {c[:110]}" for m, c in zip(meta, chunks)], hoverinfo="text")
sel = sorted(top10)
fig.add_scatter(x=xy[sel,0], y=xy[sel,1], mode="markers", name="top-10 retrieved",
                marker=dict(size=11, color="#f87171"),
                text=[f"{meta[i]['doc_id']}: {chunks[i][:110]}" for i in sel], hoverinfo="text")
fig.add_scatter(x=qxy[:,0], y=qxy[:,1], mode="markers+text", name="THE QUERY",
                marker=dict(size=18, color="#111827", symbol="star"),
                text=["  ⭐ query"], textposition="middle right")
fig.update_layout(title="Retrieval is proximity: the query lands in payment territory",
                  width=950, height=620)
fig.show()

That ⭐ is your question, living in the same space as the knowledge. **"Search" stopped being string matching and became geometry.** Everything else today — indexes, hybrid, reranking — is engineering around this one picture.

---
## 3 · 🧱 The scale wall — and how HNSW climbs it

Session 1's search was `E @ q` — touch **every** vector, every query. Honest question: how bad is that? Let's measure, then extrapolate:

In [ ]:
import time

def bench(n, dims=384, repeats=5):
    M = np.random.rand(n, dims).astype(np.float32)
    q = np.random.rand(dims).astype(np.float32)
    t0 = time.perf_counter()
    for _ in range(repeats):
        (M @ q).argmax()
    return (time.perf_counter() - t0) / repeats * 1000  # ms

for n in [len(chunks), 10_000, 100_000, 500_000]:
    print(f"{n:>9,} vectors → {bench(n):7.1f} ms per query (brute force)")

ms_500k = bench(500_000)
print(f"\nextrapolated:  5,000,000 vectors → ~{ms_500k*10:,.0f} ms per query")
print("...per query. Now multiply by queries-per-second, and add that the matrix no longer fits in RAM.")

Brute force is *linear*: 10× the data, 10× the latency, on every single query. Real systems use **ANN — Approximate Nearest Neighbor** indexes, and the dominant one is **HNSW** (*Hierarchical Navigable Small World*).

The idea is beautifully old: it's a **skip list, made of graphs**.

- Build a graph where every vector links to its nearest neighbors (bottom layer — detailed, complete).
- Stack sparser and sparser layers on top — each layer a coarse "highway network" over the one below.
- To search: start at the top layer, greedily hop toward the query, drop a layer, repeat. Like navigating by country → city → street → house number.

Each hop halves your distance-ish; you reach the neighborhood of the answer after touching a few hundred vectors out of millions — **O(log N)** instead of O(N). The price is the "A" in ANN: it's *approximate*. Occasionally the true best neighbor is missed. Every vector DB exposes knobs (`ef`, `M`) trading recall against speed — that trade-off IS the product.

In [ ]:
# A cartoon of HNSW's descent — illustrative, not the real algorithm
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
layers = [12, 40, 160]
target = np.array([0.82, 0.3])
entry  = np.array([0.08, 0.85])
for ax, n, name in zip(axes, layers, ["Layer 2 (sparse highways)", "Layer 1", "Layer 0 (all vectors)"]):
    pts = rng.random((n, 2))
    ax.scatter(pts[:,0], pts[:,1], s=14, color="#94a3b8")
    hops = np.linspace(0, 1, 4)[:, None]
    path = entry*(1-hops) + target*hops + rng.normal(0, 0.02, (4,2))
    ax.plot(path[:,0], path[:,1], "o-", color="#ef4444", lw=2, ms=6)
    ax.scatter(*target, s=140, color="#16a34a", marker="*", zorder=5)
    ax.set_title(name, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("HNSW: greedy hops on coarse layers, refine on dense ones — touch hundreds, not millions")
plt.tight_layout(); plt.show()

---
## 4 · 🗄️ A real vector database — in this notebook

**Qdrant** runs fully in-memory inside Python — zero setup, but it's the genuine article: HNSW under the hood, and the same API you'd use against a production cluster. Everything here maps 1:1 to Milvus, Weaviate, pgvector, or Pinecone — the concepts are the industry, the syntax is the vendor.

Two concepts:
- a **collection** = a table of vectors with a fixed dimensionality and distance metric
- each **point** = id + vector + **payload** (arbitrary JSON metadata). The payload is where the engineering value hides.

In [ ]:
from qdrant_client import QdrantClient, models

vdb = QdrantClient(":memory:")
vdb.create_collection("nordwind",
    vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE))

vdb.upsert("nordwind", points=[
    models.PointStruct(id=i, vector=E[i].tolist(),
                       payload={**meta[i], "text": chunks[i]})
    for i in range(len(chunks))])

print(vdb.get_collection("nordwind").points_count, "points indexed")

In [ ]:
def vsearch(query, k=5, flt=None):
    qv = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    res = vdb.query_points("nordwind", query=qv, limit=k, query_filter=flt)
    return res.points

for p in vsearch("Why did payments time out in the evening?"):
    print(f"{p.score:.3f}  {p.payload['doc_id']:12s} {p.payload['title'][:58]}")

Same results as our numpy version — as it should be. Now the payoff of payloads. Ask: *"how do I **respond right now** to a payment timeout?"* Semantically, postmortems and runbooks both match. But the user needs a **runbook** — that's not a semantic distinction, it's a *metadata* one:

In [ ]:
flt = models.Filter(must=[
    models.FieldCondition(key="doc_type", match=models.MatchValue(value="runbook"))])

print("── unfiltered (mixed intent) ──")
for p in vsearch("how to respond to payment timeouts right now", 3):
    print(f"{p.score:.3f}  [{p.payload['doc_type']:12s}] {p.payload['title'][:55]}")
print("\n── doc_type = runbook (operational intent) ──")
for p in vsearch("how to respond to payment timeouts right now", 3, flt):
    print(f"{p.score:.3f}  [{p.payload['doc_type']:12s}] {p.payload['title'][:55]}")

**Filtering is the most underrated feature in RAG.** Per-user access control, date ranges, product areas, languages — in production, almost every retrieval call carries a filter. And crucially, Qdrant filters *during* HNSW traversal, not after — post-hoc filtering of top-k can leave you with 0 results.

---
## 5 · 🔎 Where pure vectors fumble — and hybrid search

Embeddings are trained on *language*. Identifiers — `INC-2118`, `ISO 20022`, error codes, ticket numbers — are not language; to the model they're near-meaningless symbols. Ask for a specific incident by ID and watch vector search confidently return *other incidents that sound similar*, while 1970s technology (**BM25**, keyword scoring, the heart of classic search engines) nails it — provided you tokenize properly:

In [ ]:
import re
from rank_bm25 import BM25Okapi

# tokenization matters! naive .split() leaves punctuation glued to words,
# so "INC-2118," never matches "INC-2118". \w+ extraction fixes it.
tok = lambda s: re.findall(r"\w+", s.lower())
bm25 = BM25Okapi([tok(c) for c in chunks])

def bm25_top(query, k=5):
    s = bm25.get_scores(tok(query))
    return [(s[i], i) for i in np.argsort(-s)[:k]]

q = "INC-2118 root cause"
print("── vector search ──")
for p in vsearch(q, 4):
    print(f"{p.score:.3f}  {p.payload['doc_id']:12s} {p.payload['title'][:55]}")
print("\n── BM25 keyword search ──")
for s, i in bm25_top(q, 4):
    print(f"{s:6.2f}  {meta[i]['doc_id']:12s} {meta[i]['title'][:55]}")

Neither is strictly better — vectors win on paraphrase, keywords win on exact symbols. Production systems run **both** and fuse the rankings. The standard fusion is **RRF** (*Reciprocal Rank Fusion*): score each document by `1/(60+rank)` in every list and add. No tuning, no score normalization headaches — embarrassingly simple, hard to beat:

In [ ]:
def hybrid(query, k=5, pool=20):
    vec_ids = [p.id for p in vsearch(query, pool)]
    kw_ids  = [i for _, i in bm25_top(query, pool)]
    rrf = {}
    for lst in (vec_ids, kw_ids):
        for rank, i in enumerate(lst):
            rrf[i] = rrf.get(i, 0) + 1.0 / (60 + rank)
    return sorted(rrf, key=rrf.get, reverse=True)[:k]

print(f"── hybrid (RRF) for: {q!r} — best of both worlds ──")
for i in hybrid(q, 5):
    print(f"  {meta[i]['doc_id']:12s} {meta[i]['title'][:58]}")

---
## 6 · 🎯 Reranking — a second opinion, on 20 candidates

Our embedding model (a **bi-encoder**) embeds the query and the documents *separately* — that's what makes indexing millions of chunks possible, but it means query and chunk never actually *meet* until the dot product. A **cross-encoder** reads the query and a chunk **together**, attending across both — far more accurate, far too slow to run on millions.

So the production pattern: **cheap model retrieves ~20 candidates → expensive model re-reads and reorders them → keep the top 5.** Accuracy where it counts, paid only on a handful:

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")  # ~80MB

def rerank(query, candidate_ids, k=5):
    scores = reranker.predict([(query, chunks[i]) for i in candidate_ids])
    order = np.argsort(-scores)
    return [(scores[j], candidate_ids[j]) for j in order[:k]]

q2 = "what should we monitor to catch certificate problems before they hurt us?"
cands = hybrid(q2, k=20, pool=30)
print("── after rerank ──")
for s, i in rerank(q2, cands):
    print(f"{s:6.2f}  {meta[i]['doc_id']:12s} {meta[i]['title'][:58]}")

---
## 7 · 🏗️ RAG v2 — the production-shaped pipeline

Assembled: **hybrid retrieval → rerank → generate.** This is honestly close to what serious RAG products run (add caching, evals, and access-control filters and you're there):

In [ ]:
def rag_v2(question, k=5):
    cands = hybrid(question, k=20, pool=30)
    top = rerank(question, cands, k)
    context = "\n\n---\n\n".join(
        f"[Source: {meta[i]['doc_id']} — {meta[i]['title']}]\n{chunks[i]}" for _, i in top)
    print("Retrieved:", ", ".join(meta[i]['doc_id'] for _, i in top), "\n")
    return ask_claude(f"Answer using ONLY the sources below. Cite source ids. "
                      f"If insufficient, say so.\n\n{context}\n\nQuestion: {question}")

print(rag_v2("We keep having certificate-related outages. What happened before and what should we monitor?"))

Better retrieval, cleaner citations, exact-identifier robustness, intent filtering. RAG v2 is a genuinely strong system. So... one last thing. You know which question is coming. 😏

In [ ]:
HARD_QUESTION = ("Which engineers have responded to incidents affecting services "
                 "that depend on payment-gateway? List all of them.")
print(rag_v2(HARD_QUESTION, k=6))

Count the names. Compare with Session 1's ground truth (10 engineers). Better plumbing found *more* fragments — and still cannot know that "depends on payment-gateway" *means* billing-engine, still cannot know when it has found **all** qualifying incidents, still cannot **join**. We upgraded every component and the wall didn't move, because the wall isn't in the components. **It's in the geometry: points in space have no edges between them.**

## 🏁 What you learned today

- Embeddings encode *aboutness* as coordinates; cosine similarity is the ruler — and it measures aboutness, not truth
- You **saw** semantic space, and retrieval became proximity
- Brute force is O(N); **HNSW** makes it O(log N) by trading a sliver of recall
- Payload **filtering**, **hybrid** BM25+vector with RRF, and **cross-encoder reranking** — the three upgrades that separate demos from products
- And: no amount of vector engineering performs a *join*

---

<div style="background: linear-gradient(120deg,#052e16,#14532d); border-radius:16px; padding:30px 36px; color:#e2e8f0;">
<h2 style="margin:0; color:#ffffff;">⏭️ To be continued...</h2>
<p style="font-size:1.05em; margin-top:14px;">Points in space have no edges. But NordWind's knowledge <i>has</i> edges — <b>owns</b>, <b>depends-on</b>, <b>responded-to</b> — we computed with them in Session 1 without noticing.</p>
<p style="font-size:1.05em;">Next session: a database where relationships are first-class citizens. You will draw queries as ASCII art — <code>(:Service)-[:DEPENDS_ON]->(:Service)</code> — and you will see the entire company as a living, draggable network that answers the hard question in <b>one line</b>.</p>
<h3 style="color:#86efac; margin-bottom:0;">Session 3: Graph Databases — <i>your knowledge has a shape</i> 🕸️</h3>
</div>